<a href="https://colab.research.google.com/github/pop123-ux/HuggingFace-Project-Learning/blob/main/course/chapter5/slicing_and_dicing_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Time to slice and dice

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [2]:
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


In [1]:
!wget "https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip"
!unzip drugsCom_raw.zip

--2026-07-02 19:08:28--  https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘drugsCom_raw.zip’

drugsCom_raw.zip        [              <=>   ]  41.00M  8.39MB/s    in 5.7s    

2026-07-02 19:08:35 (7.22 MB/s) - ‘drugsCom_raw.zip’ saved [42989872]

Archive:  drugsCom_raw.zip
  inflating: drugsComTest_raw.tsv    
  inflating: drugsComTrain_raw.tsv   


In [1]:
from datasets import load_dataset

data_files = {'train': 'drugsComTrain_raw.tsv', 'test': 'drugsComTest_raw.tsv'}
# /t is the tab character
drug_dataset = load_dataset('csv', data_files=data_files, delimiter='\t')

In [5]:
drug_dataset

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 161297
    })
    test: Dataset({
        features: ['Unnamed: 0', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53766
    })
})

In [2]:
drug_sample = drug_dataset['train'].shuffle(seed=42).select(range(1000))
# Let's look at the first few samples
drug_sample[:3]

{'Unnamed: 0': [87571, 178045, 80482],
 'drugName': ['Naproxen', 'Duloxetine', 'Mobic'],
 'condition': ['Gout, Acute', 'ibromyalgia', 'Inflammatory Conditions'],
 'review': ['"like the previous person mention, I&#039;m a strong believer of aleve, it works faster for my gout than the prescription meds I take. No more going to the doctor for refills.....Aleve works!"',
  '"I have taken Cymbalta for about a year and a half for fibromyalgia pain. It is great\r\nas a pain reducer and an anti-depressant, however, the side effects outweighed \r\nany benefit I got from it. I had trouble with restlessness, being tired constantly,\r\ndizziness, dry mouth, numbness and tingling in my feet, and horrible sweating. I am\r\nbeing weaned off of it now. Went from 60 mg to 30mg and now to 15 mg. I will be\r\noff completely in about a week. The fibro pain is coming back, but I would rather deal with it than the side effects."',
  '"I have been taking Mobic for over a year with no side effects other than 

In [3]:
# Let's verify that the number of IDs matches the number of rows in each split:
for split in drug_dataset.keys():
  assert len(drug_dataset[split]) == len(drug_dataset[split].unique("Unnamed: 0"))

In [4]:
drug_dataset = drug_dataset.rename_column(
    original_column_name="Unnamed: 0", new_column_name="patient_id"
)

In [21]:
drug_dataset

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 161297
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53766
    })
})

In [5]:
for split in drug_dataset.keys():
  print(drug_dataset[split]['drugName'])
  print(f'{drug_dataset[split]['condition']}\n')

Column(['Valsartan', 'Guanfacine', 'Lybrel', 'Ortho Evra', 'Buprenorphine / naloxone'])
Column(['Left Ventricular Dysfunction', 'ADHD', 'Birth Control', 'Birth Control', 'Opiate Dependence'])

Column(['Mirtazapine', 'Mesalamine', 'Bactrim', 'Contrave', 'Cyclafem 1 / 35'])
Column(['Depression', "Crohn's Disease, Maintenance", 'Urinary Tract Infection', 'Weight Loss', 'Birth Control'])



In [6]:
# Let's normalize all the condition labels using Dataset.map()
def lowercase_condition(example):
  return {"condition": example['condition'].lower()}

drug_dataset.map(lowercase_condition)

Map:   0%|          | 0/161297 [00:00<?, ? examples/s]

AttributeError: 'NoneType' object has no attribute 'lower'

In [7]:
def filter_nones(x):
  return x['condition'] is not None

In [8]:
drug_dataset = drug_dataset.filter(lambda x: x['condition'] is not None)

In [9]:
drug_dataset = drug_dataset.map(lowercase_condition)
# We check if the lowercasing worked
drug_dataset['train']['condition'][:3]

Map:   0%|          | 0/160398 [00:00<?, ? examples/s]

Map:   0%|          | 0/53471 [00:00<?, ? examples/s]

['left ventricular dysfunction', 'adhd', 'birth control']

In [10]:
def compute_review_length(example):
  return {"review_length": len(example['review'].split())}

In [11]:
drug_dataset = drug_dataset.map(compute_review_length)
# Let's now inspect the first training example
drug_dataset['train'][0]

Map:   0%|          | 0/160398 [00:00<?, ? examples/s]

Map:   0%|          | 0/53471 [00:00<?, ? examples/s]

{'patient_id': 206461,
 'drugName': 'Valsartan',
 'condition': 'left ventricular dysfunction',
 'review': '"It has no side effect, I take it in combination of Bystolic 5 Mg and Fish Oil"',
 'rating': 9.0,
 'date': 'May 20, 2012',
 'usefulCount': 27,
 'review_length': 17}

In [12]:
drug_dataset['train'].sort('review_length')[:5]
# As we can see here some reviews contain only a single word
# This would be fine for sentiment analysis, however it would not be informative if we want to predict the condition

{'patient_id': [111469, 13653, 53602, 135265, 160223],
 'drugName': ['Ledipasvir / sofosbuvir',
  'Amphetamine / dextroamphetamine',
  'Alesse',
  'Zegerid',
  'Rivaroxaban'],
 'condition': ['hepatitis c',
  'adhd',
  'birth control',
  'gerd',
  'deep vein thrombosis'],
 'review': ['"Headache"', '"Great"', '"Awesome"', '"expensive"', '"Good"'],
 'rating': [10.0, 10.0, 10.0, 10.0, 9.0],
 'date': ['February 3, 2015',
  'October 20, 2009',
  'November 23, 2015',
  'March 12, 2008',
  'December 8, 2013'],
 'usefulCount': [41, 3, 0, 32, 11],
 'review_length': [1, 1, 1, 1, 1]}

In [13]:
# So we filter by length
drug_dataset = drug_dataset.filter(lambda x: x['review_length'] > 30)
drug_dataset.num_rows

Filter:   0%|          | 0/160398 [00:00<?, ? examples/s]

Filter:   0%|          | 0/53471 [00:00<?, ? examples/s]

{'train': 138514, 'test': 46108}

In [14]:
# Now lets deal with the presence of HTML character codes in the reviews
import html

text = "I&#039;m a transformer called BERT"
html.unescape(text)

"I'm a transformer called BERT"

In [16]:
drug_dataset = drug_dataset.map(lambda x: {'review': html.unescape(x['review'])})

Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

In [18]:
# To tokenize all the drug reviews with a fast tokenizer, we could use a function like this:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

def tokenize_function(examples):
  return tokenizer(examples['review'], truncation=True)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [19]:
%time tokenized_dataset = drug_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

CPU times: user 2min, sys: 666 ms, total: 2min 1s
Wall time: 1min 38s


In [21]:
slow_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=False)


def slow_tokenize_function(examples):
    return slow_tokenizer(examples["review"], truncation=True)


tokenized_dataset = drug_dataset.map(slow_tokenize_function, batched=True, num_proc=8)

Map (num_proc=8):   0%|          | 0/138514 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/46108 [00:00<?, ? examples/s]

In [24]:
# Now let's tokenize the examples and truncate them to a max length of 128,
# We will ask the tokenizer to return all the chunks of the texts instead of just the first one
def tokenize_and_split(example):
  return tokenizer(
      example['review'],
      truncation=True,
      max_length=128,
      return_overflowing_tokens=True,
  )
# Our first example in the training set became two features, since it was tokenized to more than the maximum number of tokens we specified: the first one of length 128 and the second one of length 49

In [25]:
result = tokenize_and_split(drug_dataset['train'][0])
[len(inp) for inp in result['input_ids']]

[128, 49]

In [26]:
tokenized_dataset = drug_dataset.map(tokenize_and_split, batched=True)

Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

ArrowInvalid: Column 8 named input_ids expected length 1000 but got length 1463

In [27]:
tokenized_dataset = drug_dataset.map(
    tokenize_and_split, batched=True, remove_columns=drug_dataset['train'].column_names
)

Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

In [28]:
len(tokenized_dataset['train']), len(drug_dataset['train'])

(206772, 138514)

In [ ]:
def tokenize_and_split(examples):
    result = tokenizer(
        examples["review"],
        truncation=True,
        max_length=128,
        return_overflowing_tokens=True,
    )

    # Extract mapping between new and old indices
    sample_map = result.pop("overflow_to_sample_mapping")
    for key, values in examples.items():
        result[key] = [values[i] for i in sample_map]
    return result

In [29]:
tokenized_dataset = drug_dataset.map(tokenize_and_split, batched=True)
tokenized_dataset

Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

ArrowInvalid: Column 8 named input_ids expected length 1000 but got length 1463

In [31]:
# Now let's try to convert the Dataset into a pandas dataframe
drug_dataset.set_format('pandas')


In [32]:
drug_dataset['train'][:3]

,patient_id,drugName,condition,review,rating,date,usefulCount,review_length
0,95260,Guanfacine,adhd,"""My son is halfway through his fourth week of ...",8.0,"April 27, 2010",192,141
1,92703,Lybrel,birth control,"""I used to take another oral contraceptive, wh...",5.0,"December 14, 2009",17,134
2,138000,Ortho Evra,birth control,"""This is my first time using any form of birth...",8.0,"November 3, 2015",10,89


In [33]:
# Let's create a pandas.DataFrame for the whole training set by selecting all the elements of drug_dataset['train']:
train_df = drug_dataset['train'][:]

In [34]:
frequencies = (
    train_df['condition']
    .value_counts()
    .to_frame()
    .reset_index()
    .rename(columns={'index': 'condition', 'count': 'frequency'})
)
frequencies.head()

,condition,frequency
0,birth control,27655
1,depression,8023
2,acne,5209
3,anxiety,4991
4,pain,4744


In [35]:
# Once we're done with our Pandas analysis, we can always create a new Dataset object by using the Dataset.from_pandas() function
from datasets import Dataset

freq_dataset = Dataset.from_pandas(frequencies)
freq_dataset

Dataset({
    features: ['condition', 'frequency'],
    num_rows: 819
})

In [45]:
S1 = train_df.groupby('drugName')['rating'].mean()

In [51]:
new_dataset = Dataset.from_pandas(S1.reset_index())
new_dataset

Dataset({
    features: ['drugName', 'rating'],
    num_rows: 3052
})

In [52]:
# Now let's create a validation set to prepare the dataset for training a classifier on
# Before that, we need to reset the output format of drug_dataset from 'pandas' to 'arrow'
drug_dataset.reset_format()

In [53]:
drug_dataset_clean = drug_dataset['train'].train_test_split(train_size=0.8, seed=42)
# Rename the default "test" split to "validation"
drug_dataset_clean['validation'] = drug_dataset_clean.pop('test')
# Add the "test" set to the 'DatasetDict'
drug_dataset_clean['test'] = drug_dataset['test']
drug_dataset_clean

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 110811
    })
    validation: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 27703
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46108
    })
})

## Here are a few ideas that you could try out: ##

- Train a classifier that can predict the patient condition based on the drug review.

- Use the summarization pipeline from to generate summaries of the reviews.